# IslamicEval — Corpus Preprocessing & Paper Artifacts (standalone, resume-safe)

Cleans the **Quran** and **Hadith** corpora exactly along your 2025 paper's pipeline
(Appendix D + E), removes **تشكيل / tatweel** so segmented lookup search works, writes a proper
field structure into your Drive, and produces **every CSV table/artifact** you need for the write-up.

**Why this won't make you restart from scratch:** every stage is a *checkpoint*. Each stage writes
its output to Drive and records `done` in a `_state.json` manifest. Re-running the notebook
**skips finished stages** and reloads their outputs. The heavy stage (overlapping-segment KB, which
explodes to millions of rows and is your OOM risk) streams to **sharded CSVs on disk** — it never
holds the full set in RAM, and it resumes from the last finished source text if the runtime dies.

**Stages**

| # | Stage | Output | Resumable |
|---|-------|--------|-----------|
| 1 | Load + validate raw corpora | `quran_clean.csv`, `hadith_clean.csv` | skip-if-done |
| 2 | Segment long verses (>25 tok) | `quran_segmented.csv` | skip-if-done |
| 3 | Diacritic augmentation (كeep + strip) | `*_augmented.csv` | skip-if-done |
| 4 | Enhanced KB (overlapping segments) | `kb_*/shard_*.csv` | **mid-stage** (per source text) |
| 5 | Optional global dedup | `kb_*_dedup.csv` | skip-if-done |
| 6 | Paper tables | `tables/*.csv` | always cheap |
| 7 | Optional figures | `figures/*.png` | always cheap |

Set the paths in **Cell 3** and run top-to-bottom. Crash? Just run it again.


## 0 · Setup

In [ ]:
!pip -q install pandas numpy tqdm
# AraBERTv2 tokenizer is used ONLY for the >25-token split rule (to match the paper).
# It downloads a tiny tokenizer, no model weights. If it fails we fall back to word counting.
!pip -q install transformers >/dev/null 2>&1 || echo "transformers optional"
print("ok")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Configuration

Point `RAW_QURAN` / `RAW_HADITH` at your source files and `OUT_DIR` at where you want the
processed corpus to live in Drive. Everything else has paper-faithful defaults.

In [ ]:
from pathlib import Path

# ---- INPUT (your raw source files) ----
RAW_QURAN  = "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/quranic_verses.json"
RAW_HADITH = "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/six_hadith_books.json"

# ---- OUTPUT (processed corpus + artifacts live here, in Drive so they survive restarts) ----
OUT_DIR = "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/processed"

# ---- preprocessing params (paper Appendix D/E) ----
MAX_TEXT_CHARS   = 1500     # length threshold to prevent memory overflow (D.1)
SPLIT_TOKEN_LEN  = 25       # verses longer than this are bisected (D.2.1)
KEEP_DIACRITICS  = True     # keep original AND add a diacritic-free copy (D.2.2)
NORM_LETTERS     = True     # also unify alef/ya/ta-marbuta for the normalized copy

# ---- enhanced KB for segmented search (Appendix E) ----
KB_MIN_WORDS   = 5          # shortest segment
KB_MAX_WORDS   = 15         # longest segment
KB_LEN_STEP    = 3          # step between segment lengths (E.1.3)
KB_WINDOW_STEP = 3          # slide step across positions (raise to shrink KB / lower RAM)
SHARD_ROWS     = 200_000    # rows per shard CSV (flush cadence -> caps RAM use)

# ---- run control ----
FORCE_REDO   = []           # e.g. ["stage4_kb"] to force-rebuild a stage; [] = resume normally
GLOBAL_DEDUP = True         # run stage 5 (streamed hash dedup of the KB)
MAKE_FIGURES = True

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)
for k in ("RAW_QURAN","RAW_HADITH"):
    print(("FOUND " if Path(eval(k)).exists() else "MISSING"), eval(k))
print("OUT_DIR ->", OUT_DIR)

## 2 · Resume manager

A tiny manifest (`_state.json` in `OUT_DIR`) records which stages finished and their row counts.
`should_run()` returns `False` when a stage is already done and its files exist — so a re-run is a
fast no-op that just reloads. Put a stage name in `FORCE_REDO` to rebuild it.

In [ ]:
import json, time

STATE_PATH = OUT / "_state.json"

def load_state():
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text(encoding="utf-8"))
    return {}

def save_state(st):
    STATE_PATH.write_text(json.dumps(st, ensure_ascii=False, indent=2), encoding="utf-8")

def should_run(name, outputs):
    st = load_state()
    done = st.get(name, {}).get("done") and all(Path(o).exists() for o in outputs)
    if done and name not in FORCE_REDO:
        print(f"[skip] {name} — already done ({st[name].get('rows','?')} rows)")
        return False
    print(f"[run ] {name}")
    return True

def mark_done(name, **meta):
    st = load_state(); st[name] = {"done": True, "ts": time.time(), **meta}; save_state(st)

print("state:", load_state())

## 3 · Arabic normalization (the تشكيل removal you asked for)

`strip_diacritics` removes all harakat, the superscript alef, and tatweel (Unicode
`U+064B–U+0652`, `U+0670`, `U+0640`) — this is what lets an undiacritized quote match the canonical
verse during segmented search. `normalize` optionally also unifies alef/ya/ta-marbuta and collapses
non-letters, matching the paper's normalization.

In [ ]:
import re

# Full Quranic diacritics + annotation marks (not just the 8 basic harakat), so segmented
# search works on Uthmani-script verses: tanwin, harakat, shadda, sukun, dagger alef, maddah,
# hamza marks, and the Quranic annotation signs (small high seen/meem, sajdah, waqf marks...).
_TASHKEEL = re.compile(
    r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]'
)
_TATWEEL  = '\u0640'
_NON_AR   = re.compile(r'[^\u0621-\u064A\s]')
_SPACES   = re.compile(r'\s+')

def strip_diacritics(text):
    if not text: return ""
    return _SPACES.sub(' ', _TASHKEEL.sub('', str(text)).replace(_TATWEEL, '')).strip()

def normalize(text, letters=True):
    t = strip_diacritics(text)
    if letters:
        t = re.sub('[إأآٱ\u0671]', 'ا', t).replace('ى','ي').replace('ؤ','و').replace('ئ','ي').replace('ة','ه')
        t = _SPACES.sub(' ', _NON_AR.sub(' ', t)).strip()
    return t

# sanity check
demo = "إِنَّاۤ أَعْطَيْنَاكَ ٱلْكَوْثَرَ"
print("raw   :", demo)
print("strip :", strip_diacritics(demo))
print("norm  :", normalize(demo))

## 4 · Stage 1 — load, validate, and structure the raw corpora

In [ ]:
import pandas as pd

def _read_json_any(path):
    p = Path(path)
    if not p.exists(): return None
    txt = p.read_text(encoding="utf-8").strip()
    if not txt: return []
    try: return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(ln) for ln in txt.splitlines() if ln.strip()]

def _pick(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ""): return d[k]
    return None

def _synth_quran():
    return [{"surah_id":1,"surah_name":"الفاتحة","ayah_id":i+1,"ayah_text":t} for i,t in enumerate([
        "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ","الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ",
        "الرَّحْمَٰنِ الرَّحِيمِ","مَالِكِ يَوْمِ الدِّينِ",
        "وَمَا خَلَقْتُ الْجِنَّ وَالْإِنسَ إِلَّا لِيَعْبُدُونِ مِنْ رِزْقٍ وَمَا أُرِيدُ أَن يُطْعِمُونِ إِنَّ اللَّهَ هُوَ الرَّزَّاقُ ذُو الْقُوَّةِ الْمَتِينُ"])]

def _synth_hadith():
    return [{"hadithID":1,"title":"البخاري","Matn":"إنما الأعمال بالنيات وإنما لكل امرئ ما نوى","isnad":"حدثنا الحميدي"},
            {"hadithID":2,"title":"مسلم","Matn":"من حسن إسلام المرء تركه ما لا يعنيه","isnad":""}]

STAGE1_OUT = [OUT/"quran_clean.csv", OUT/"hadith_clean.csv"]
if should_run("stage1_load", STAGE1_OUT):
    qd = _read_json_any(RAW_QURAN)  or (print("[quran] SYNTHETIC") or _synth_quran())
    hd = _read_json_any(RAW_HADITH) or (print("[hadith] SYNTHETIC") or _synth_hadith())

    q_rows = []
    for d in qd:
        raw = _pick(d, ["ayah_text","full_text","span_text","text"])
        if not raw or len(str(raw)) >= MAX_TEXT_CHARS:   # length threshold (D.1)
            continue
        q_rows.append({"surah_id":_pick(d,["surah_id","surah","surahId"]),
                       "surah_name":_pick(d,["surah_name","surahName"]),
                       "ayah_id":_pick(d,["ayah_id","ayahId","verse_id"]),
                       "text_raw":str(raw)})
    q = pd.DataFrame(q_rows).drop_duplicates("text_raw").reset_index(drop=True)

    h_rows = []
    for d in hd:
        matn = _pick(d, ["Matn","matn","hadithTxt","hadith_text","text"])
        if not matn or len(str(matn)) >= MAX_TEXT_CHARS:   # filter empty matn + length (D.1)
            continue
        h_rows.append({"hadith_id":_pick(d,["hadithID","hadith_id","id"]),
                       "book":_pick(d,["title","book","BookName"]),
                       "book_id":_pick(d,["BookID","book_id"]),
                       "isnad_raw":_pick(d,["isnad","sanad","chain"]) or "",
                       "text_raw":str(matn)})
    h = pd.DataFrame(h_rows).drop_duplicates("text_raw").reset_index(drop=True)

    q.to_csv(STAGE1_OUT[0], index=False)
    h.to_csv(STAGE1_OUT[1], index=False)
    mark_done("stage1_load", rows=int(len(q)+len(h)), quran=int(len(q)), hadith=int(len(h)))

QURAN_CLEAN  = pd.read_csv(STAGE1_OUT[0]).fillna("")
HADITH_CLEAN = pd.read_csv(STAGE1_OUT[1]).fillna("")
print(f"quran_clean={len(QURAN_CLEAN)}  hadith_clean={len(HADITH_CLEAN)}")
QURAN_CLEAN.head(3)

## 5 · Stage 2 — split verses longer than 25 tokens (paper D.2.1)

Long verses are bisected at the whitespace nearest the midpoint (content-aware split, no word is
broken), limited to two parts. Token length uses the AraBERTv2 tokenizer to match the paper, with a
whitespace fallback if `transformers` isn't available.

In [ ]:
try:
    from transformers import AutoTokenizer
    _TOK = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
    def n_tokens(t): return len(_TOK.tokenize(t))
    print("using AraBERTv2 tokenizer")
except Exception as e:
    print("tokenizer unavailable -> whitespace fallback:", e)
    def n_tokens(t): return len(t.split())

def split_long(text):
    if n_tokens(text) <= SPLIT_TOKEN_LEN:
        return [text]
    words = text.split()
    mid = len(words)//2                    # approximate midpoint, search nearest boundary
    return [" ".join(words[:mid]).strip(), " ".join(words[mid:]).strip()]

STAGE2_OUT = [OUT/"quran_segmented.csv"]
if should_run("stage2_segment", STAGE2_OUT):
    rows = []
    for _, r in QURAN_CLEAN.iterrows():
        for i, seg in enumerate(split_long(r["text_raw"])):
            if seg:
                rows.append({**r.to_dict(), "text_raw":seg, "seg_part":i})
    seg = pd.DataFrame(rows).reset_index(drop=True)
    seg.to_csv(STAGE2_OUT[0], index=False)
    mark_done("stage2_segment", rows=int(len(seg)), from_verses=int(len(QURAN_CLEAN)))

QURAN_SEG = pd.read_csv(STAGE2_OUT[0]).fillna("")
print(f"verses {len(QURAN_CLEAN)} -> segments {len(QURAN_SEG)}")

## 6 · Stage 3 — diacritic augmentation (paper D.2.2)

For every Quran segment we keep the original and add a **diacritic-free** normalized copy
(`variant` = `raw` / `norm`), which is what doubles the effective corpus and makes matching robust to
vocalization. Hadith get the same treatment. This produces the proper structured fields in Drive.

In [ ]:
def augment(df, text_col="text_raw"):
    out = []
    for _, r in df.iterrows():
        d = r.to_dict()
        out.append({**d, "variant":"raw",  "text":d[text_col], "text_norm":normalize(d[text_col], NORM_LETTERS)})
        if KEEP_DIACRITICS:
            stripped = strip_diacritics(d[text_col])
            out.append({**d, "variant":"norm", "text":stripped, "text_norm":normalize(stripped, NORM_LETTERS)})
    return pd.DataFrame(out)

STAGE3_OUT = [OUT/"quran_augmented.csv", OUT/"hadith_augmented.csv"]
if should_run("stage3_augment", STAGE3_OUT):
    qa = augment(QURAN_SEG).drop_duplicates("text_norm").reset_index(drop=True)
    ha = augment(HADITH_CLEAN).drop_duplicates("text_norm").reset_index(drop=True)
    qa.to_csv(STAGE3_OUT[0], index=False)
    ha.to_csv(STAGE3_OUT[1], index=False)
    mark_done("stage3_augment", rows=int(len(qa)+len(ha)), quran=int(len(qa)), hadith=int(len(ha)))

QURAN_AUG  = pd.read_csv(STAGE3_OUT[0]).fillna("")
HADITH_AUG = pd.read_csv(STAGE3_OUT[1]).fillna("")
print(f"quran_aug={len(QURAN_AUG)}  hadith_aug={len(HADITH_AUG)}")
QURAN_AUG.head(4)

## 7 · Stage 4 — enhanced KB of overlapping segments (Appendix E) — **streamed & mid-stage resumable**

This is the stage that used to blow up your RAM: every verse/hadith is exploded into overlapping
`KB_MIN_WORDS…KB_MAX_WORDS`-word windows. Instead of building a giant list, we **stream rows to
sharded CSVs** and record the index of the last finished source text in `_state.json`. If the
runtime dies at text 18,000 of 30,000, the next run continues from 18,000 — not from zero. Raise
`KB_WINDOW_STEP` / `SHARD_ROWS` to trade recall for lower memory and fewer files.

In [ ]:
import csv, glob

def overlapping_segments(text_norm):
    words = text_norm.split()
    W = len(words)
    seen = set()
    # always include the whole (normalized) text
    if W: 
        yield text_norm
        seen.add(text_norm)
    for L in range(KB_MIN_WORDS, KB_MAX_WORDS + 1, KB_LEN_STEP):
        if L >= W:                      # whole text already covers it
            break
        for s in range(0, W - L + 1, KB_WINDOW_STEP):
            seg = " ".join(words[s:s+L])
            if seg not in seen:
                seen.add(seg); yield seg

def build_kb(df, name):
    '''Stream overlapping segments of df['text_norm'] into OUT/name/shard_*.csv, resumable.'''
    kb_dir = OUT / name; kb_dir.mkdir(exist_ok=True)
    st = load_state()
    prog = st.get(name, {})
    if prog.get("done") and name not in FORCE_REDO:
        print(f"[skip] {name} — {prog.get('rows','?')} segments"); return kb_dir
    start_idx = prog.get("next_idx", 0) if name not in FORCE_REDO else 0
    shard_idx = prog.get("shard_idx", 0)
    total     = prog.get("rows", 0)
    if name in FORCE_REDO:
        for f in glob.glob(str(kb_dir/"shard_*.csv")): Path(f).unlink()
        start_idx = shard_idx = total = 0

    buf, buf_seen = [], set()
    def flush():
        nonlocal shard_idx, total, buf, buf_seen
        if not buf: return
        path = kb_dir / f"shard_{shard_idx:05d}.csv"
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(["src_id","segment"]); w.writerows(buf)
        total += len(buf); shard_idx += 1; buf, buf_seen = [], set()

    from tqdm.auto import tqdm
    texts = df["text_norm"].tolist()
    try:
        for i in tqdm(range(start_idx, len(texts)), initial=start_idx, total=len(texts)):
            for seg in overlapping_segments(texts[i]):
                key = (i, seg)
                if seg in buf_seen:   # cheap intra-buffer dedup
                    continue
                buf_seen.add(seg); buf.append([i, seg])
                if len(buf) >= SHARD_ROWS:
                    flush()
                    # checkpoint AFTER a clean flush so resume is exact
                    mark_done_partial(name, next_idx=i+1, shard_idx=shard_idx, rows=total)
        flush()
        mark_done(name, rows=int(total), shards=int(shard_idx), source_rows=int(len(texts)))
    except (KeyboardInterrupt, MemoryError) as e:
        flush(); mark_done_partial(name, next_idx=i, shard_idx=shard_idx, rows=total)
        print(f"[interrupted] {name} at text {i}; progress saved — just re-run this cell. ({e})")
        raise
    return kb_dir

def mark_done_partial(name, **meta):
    st = load_state(); cur = st.get(name, {}); cur.update({"done": False, **meta}); st[name]=cur; save_state(st)

KB_Q = build_kb(QURAN_AUG,  "kb_quran_segments")
KB_H = build_kb(HADITH_AUG, "kb_hadith_segments")
print("KB quran shards:", len(glob.glob(str(KB_Q/'shard_*.csv'))),
      "| KB hadith shards:", len(glob.glob(str(KB_H/'shard_*.csv'))))

## 8 · Stage 5 — optional global dedup of the KB (streamed, low-memory)

Per-shard dedup already ran during Stage 4. This optional pass removes duplicates *across* shards by
streaming every shard and keeping an 8-byte hash set (a few million segments ≈ tens of MB — safe).
Writes one consolidated `kb_*_dedup.csv` per corpus.

In [ ]:
import hashlib, glob

def dedup_kb(kb_dir, out_csv, name):
    if should_run(name, [out_csv]) is False:
        return
    seen = set(); kept = 0
    with open(out_csv, "w", newline="", encoding="utf-8") as fo:
        w = csv.writer(fo); w.writerow(["segment"])
        for shard in sorted(glob.glob(str(kb_dir/"shard_*.csv"))):
            with open(shard, encoding="utf-8") as fi:
                r = csv.reader(fi); next(r, None)
                for row in r:
                    seg = row[1] if len(row) > 1 else row[0]
                    h = hashlib.blake2b(seg.encode("utf-8"), digest_size=8).digest()
                    if h in seen: continue
                    seen.add(h); w.writerow([seg]); kept += 1
    mark_done(name, rows=int(kept))
    print(f"[{name}] unique segments: {kept}")

if GLOBAL_DEDUP:
    dedup_kb(OUT/"kb_quran_segments",  OUT/"kb_quran_dedup.csv",  "stage5_dedup_quran")
    dedup_kb(OUT/"kb_hadith_segments", OUT/"kb_hadith_dedup.csv", "stage5_dedup_hadith")
else:
    print("GLOBAL_DEDUP=False -> skipped")

## 9 · Stage 6 — paper tables (CSV artifacts)

Regenerates the paper's descriptive tables from the actual processed files (so the numbers always
match what's on disk): source counts (Table 1b), post-preprocessing counts (Table 1c), and
length statistics for verses and hadith (Table 5b-style: count/mean/std/min/max).

In [ ]:
import numpy as np

def len_stats(series, label):
    L = series.astype(str).str.len()
    return {"corpus":label, "count":int(len(L)), "mean":round(float(L.mean()),1),
            "std":round(float(L.std()),1), "min":int(L.min()), "max":int(L.max())}

def _kb_count(name, dedup_csv, kb_dir):
    st = load_state().get(name, {})
    if st.get("rows") is not None: return st["rows"]
    if Path(dedup_csv).exists(): return sum(1 for _ in open(dedup_csv, encoding="utf-8")) - 1
    return sum(sum(1 for _ in open(s, encoding="utf-8"))-1 for s in glob.glob(str(kb_dir/"shard_*.csv")))

# Table 1b — original source counts
t1b = pd.DataFrame([
    {"corpus":"Quranic Verses (Ayahs)", "original_count":int(len(QURAN_CLEAN))},
    {"corpus":"Hadith Narrations",      "original_count":int(len(HADITH_CLEAN))},
    {"corpus":"Total Unique Texts",     "original_count":int(len(QURAN_CLEAN)+len(HADITH_CLEAN))},
])
# Table 1c — after preprocessing (segmented + diacritic-augmented uniques)
t1c = pd.DataFrame([
    {"corpus":"Total Unique Ayahs",  "preprocessed_count":int(len(QURAN_AUG))},
    {"corpus":"Total Unique Hadiths","preprocessed_count":int(len(HADITH_AUG))},
    {"corpus":"Total Unique Texts",  "preprocessed_count":int(len(QURAN_AUG)+len(HADITH_AUG))},
])
# Table (length statistics)
t_len = pd.DataFrame([len_stats(QURAN_AUG["text"], "Ayah"), len_stats(HADITH_AUG["text"], "Hadith")])
# KB size table
t_kb = pd.DataFrame([
    {"kb":"Quran segments",  "count":int(_kb_count("stage5_dedup_quran",  OUT/"kb_quran_dedup.csv",  OUT/"kb_quran_segments"))},
    {"kb":"Hadith segments", "count":int(_kb_count("stage5_dedup_hadith", OUT/"kb_hadith_dedup.csv", OUT/"kb_hadith_segments"))},
])

for df_, fn in [(t1b,"table_source_counts.csv"),(t1c,"table_preprocessed_counts.csv"),
                (t_len,"table_length_stats.csv"),(t_kb,"table_kb_sizes.csv")]:
    df_.to_csv(OUT/"tables"/fn, index=False)
mark_done("stage6_tables", files=4)
print("Table 1b (source counts):");        print(t1b.to_string(index=False))
print("\nTable 1c (preprocessed):");        print(t1c.to_string(index=False))
print("\nLength stats:");                   print(t_len.to_string(index=False))
print("\nKB sizes:");                       print(t_kb.to_string(index=False))

## 10 · Stage 7 — optional figures for the paper

In [ ]:
if MAKE_FIGURES:
    import matplotlib.pyplot as plt
    # verse/hadith length distributions
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    QURAN_AUG["text"].astype(str).str.len().hist(bins=40, ax=ax[0]); ax[0].set_title("Ayah length (chars)")
    HADITH_AUG["text"].astype(str).str.len().hist(bins=40, ax=ax[1]); ax[1].set_title("Hadith matn length (chars)")
    for a in ax: a.set_xlabel("characters"); a.set_ylabel("count")
    fig.tight_layout(); fig.savefig(OUT/"figures"/"length_distributions.png", dpi=150)
    plt.show()
    # corpus growth bar
    fig2, ax2 = plt.subplots(figsize=(6,4))
    ax2.bar(["Quran raw","Quran aug","Hadith raw","Hadith aug"],
            [len(QURAN_CLEAN),len(QURAN_AUG),len(HADITH_CLEAN),len(HADITH_AUG)])
    ax2.set_title("Corpus size before/after preprocessing"); ax2.set_ylabel("unique texts")
    fig2.tight_layout(); fig2.savefig(OUT/"figures"/"corpus_growth.png", dpi=150); plt.show()
    mark_done("stage7_figures", files=2)
    print("figures saved ->", OUT/"figures")
else:
    print("MAKE_FIGURES=False -> skipped")

## 11 · Manifest & artifact inventory

In [ ]:
import glob, os
print("=== _state.json ===")
print(json.dumps(load_state(), ensure_ascii=False, indent=2))
print("\n=== artifacts in", OUT_DIR, "===")
for p in sorted(glob.glob(str(OUT/"**"/"*"), recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1024:9.1f} KB  {os.path.relpath(p, OUT)}")
print("\nDone. Re-running this notebook will SKIP every finished stage above.")

## 12 · How resume works (quick reference)

- **Whole-stage skip:** stages 1–3, 5–7 check `_state.json`; if `done` and files exist, they reload
  instead of recomputing.
- **Mid-stage resume (Stage 4):** after every shard flush it records `next_idx` (the next source
  text to process). A crash → re-run continues from there. Delete nothing.
- **Force a rebuild:** add the stage name to `FORCE_REDO` in Cell 1, e.g.
  `FORCE_REDO = ["stage4_kb_quran_segments"]`, and re-run.
- **Lower memory further:** raise `KB_WINDOW_STEP` (fewer overlapping windows) and lower `SHARD_ROWS`
  (more frequent flushes → smaller RAM buffer).
- **Outputs live in Drive**, so a disconnected/OOM runtime never loses finished work.
